#### Full Fine Tuning

Full Fine Tuning is very time consuming process. Here we are going to use very small LLM to understand the concept.

##### 1. Understanding Small Model for Fine Tuning

In [1]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "false"
import torch
from transformers import pipeline

d:\ProjectDesk\gh-iamatulkumar-ya\data-science\.venv_llm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# using very small model
model_id = "arnir0/Tiny-LLM"

In [3]:
tiny_llm_model = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 12/12 [00:00<00:00, 11998.01it/s]


In [4]:
query="Happiness is not something ready made. It comes from"

In [5]:
tiny_llm_model(query)[0]["generated_text"]

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


'Happiness is not something ready made. It comes from the floor and we are now.\n|11:00 a. This is our first time in our world, all of our own. We are a few times in the world.\n11.7.27\nSeveral more than 1500,000 people in the world.\n27571 1990.'

Perfect superfast, let's finetune with our Quotes data and see what response it gives once fully fine tuned

##### 2. Loading Model Configs

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

In [7]:
# loading model tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

In [8]:
# loading base model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype=torch.bfloat16

)

Loading weights: 100%|██████████| 12/12 [00:00<00:00, 4001.24it/s]


##### 3. Defining SFTConfig and Trainer for Training

In [13]:
# adding sft config

sft_config = SFTConfig(
    output_dir="./tiny-llm-fft-training",  # The file path where the model checkpoints, tokenizer files, and training logs will be saved.
    dataset_text_field="text", # Tells the trainer which specific column name in your dataset contains the text to be trained on.
    packing=False, # If true, it combines multiple short examples into a single sequence of
    per_device_train_batch_size=4, # The number of training examples processed simultaneously on a single GPU.
    gradient_accumulation_steps=4, # The number of steps to wait (accumulating gradients) before performing a single weight update, effectively increasing the "total" batch size.
    learning_rate=2e-5, # The "step size" the optimizer takes to minimize the error; too high may cause instability, too low may be too slow.
    num_train_epochs=1, # The total number of times the model will see the entire training dataset.
    save_steps=100, # How often (in training steps) the trainer saves a backup checkpoint of the model to the output directory.
    logging_steps=10, # How frequently the training progress (loss, learning rate, etc.) is printed to the console or log.
    lr_scheduler_type="cosine", # Defines how the learning rate changes over time (e.g., "cosine" smoothly reduces the rate towards the end of training).
    #optim="paged_adamw_32bit", # The specific optimization algorithm used; paged_adamw_32bit is memory-efficient and ideal for QLoRA.
    report_to="none", # Specifies external platforms (like Weights & Biases or TensorBoard) where training metrics should be sent for visualization.
    use_cpu=True,
)

In [10]:
# laoding dataset
from datasets import Dataset
dataset = Dataset.from_text(os.path.join(os.getcwd(), "full_fine_tuning_training_data.jsonl"))


In [14]:
# initilizing SFTTrainer

trainer = SFTTrainer(
    model=model_id,
    args=sft_config,
    train_dataset=dataset 
)

Loading weights: 100%|██████████| 12/12 [00:00<00:00, 12000.87it/s]


In [16]:
# let's see how many paramter trainer will be fine tuning
trainer.get_num_trainable_parameters()

12988992

In [ ]:
# total model paramters
model.num_parameters()

12988992

as this is full fine tuning, all paramters will get updated,

##### 4. Train and Save

In [15]:
# training the model
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,5.060301
20,4.138134
30,3.672079
40,3.371397
50,3.267974
60,3.194955


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 23.26it/s]


TrainOutput(global_step=60, training_loss=3.7841399192810057, metrics={'train_runtime': 1780.9417, 'train_samples_per_second': 0.535, 'train_steps_per_second': 0.034, 'total_flos': 2226155678208.0, 'train_loss': 3.7841399192810057})

In [23]:
# saving the model
trainer.save_model("./tiny-llm-fft-model")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 21.78it/s]


##### 5. Let's Load and Check Fine Tuned Model

In [24]:
# loading fully fine tuned model
ft_model = AutoModelForCausalLM.from_pretrained("./tiny-llm-fft-model")

Loading weights: 100%|██████████| 12/12 [00:00<00:00, 12018.06it/s]


In [26]:
# let's check the response
input = f"Quote: {query} | Author: | Source: | Completion:"
input_token = tokenizer(input, return_tensors="pt")["input_ids"]

output = ft_model.generate(input_token, max_new_tokens=60)
tokenizer.batch_decode(output)

["<s> Quote: Happiness is not something ready made. It comes from | Author: | Source: | Completion: 'the world' | Author: 'the world of the world.'eness is the world' | Author: 'the world of the world.'eness is the world' | Author: 'the world of the world.'comments powered by\nthe world' | Author: 'the world of the"]

Hmm, not impressing but still able to return what we were a bit expecting.

Accuracy and performance depends upon the model and the training. As we did this training on local machine thus had to reduce some parameters which cuases this behaviour.